In [ ]:
import tkinter as tk
import heapq, math, random, time

ROWS, COLS, CELL = 15, 20, 30 # rows columns pixelSize
E, W, S, G, F, V, P, A = range(8)   # Empty Wall Start Goal Frontier Visited Path Agent

COLOR = { E:"#1e1e2e", W:"#585b70", S:"#89b4fa", G:"#f38ba8",
          F:"#f9e2af",  V:"#3b82f6", P:"#a6e3a1", A:"#cba6f7" }

# ── Heuristics ────────────────────────────────────────────────────────────────
def manhattanDist(a, b): return abs(a[0]-b[0]) + abs(a[1]-b[1])
def euclideanDist(a, b): return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)

# ── Grid helpers ──────────────────────────────────────────────────────────────
def getNeighbours(grid, r, c):
    for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
        nr, nc = r+dr, c+dc
        if 0 <= nr < ROWS and 0 <= nc < COLS and grid[nr][nc] != W:
            yield (nr, nc)

def buildPath(cameFrom, goal):
    path, node = [], goal
    while node in cameFrom: path.append(node); node = cameFrom[node]
    path.append(node); path.reverse(); return path

# ── Search algorithms (return animation steps) ────────────────────────────────
def runAstar(grid, start, goal, hFn):
    openSet, cameFrom, gScore, visited, steps = [], {}, {start: 0}, set(), []
    heapq.heappush(openSet, (0, 0, start))
    while openSet:
        _, g, cur = heapq.heappop(openSet)
        if cur in visited: continue
        visited.add(cur)
        steps.append((frozenset(item[2] for item in openSet), frozenset(visited), None))
        if cur == goal:
            steps.append((frozenset(), frozenset(visited), buildPath(cameFrom, goal))); return steps
        for nb in getNeighbours(grid, *cur):
            newG = g + 1
            if newG < gScore.get(nb, float("inf")):
                gScore[nb] = newG; cameFrom[nb] = cur
                heapq.heappush(openSet, (newG + hFn(nb, goal), newG, nb))
    steps.append((frozenset(), frozenset(visited), [])); return steps

def runGbfs(grid, start, goal, hFn):
    counter, openSet, cameFrom, visited, inF, steps = 0, [(hFn(start,goal),0,start)], {}, set(), {start}, []
    heapq.heapify(openSet)
    while openSet:
        _, _, cur = heapq.heappop(openSet)
        if cur in visited: continue
        inF.discard(cur); visited.add(cur)
        steps.append((frozenset(inF), frozenset(visited), None))
        if cur == goal:
            steps.append((frozenset(), frozenset(visited), buildPath(cameFrom, goal))); return steps
        for nb in getNeighbours(grid, *cur):
            if nb not in visited and nb not in inF:
                cameFrom[nb] = cur; counter += 1
                heapq.heappush(openSet, (hFn(nb,goal), counter, nb)); inF.add(nb)
    steps.append((frozenset(), frozenset(visited), [])); return steps

def quickPath(grid, start, goal, hFn, algoFn):
    steps = algoFn(grid, start, goal, hFn)
    return steps[-1][2] if steps else []

# ── App ───────────────────────────────────────────────────────────────────────
class App:
    def __init__(self, root):
        self.root = root
        self.root.title("Dynamic Pathfinding Agent")
        self.root.configure(bg="#11111b")

        self.grid      = [[E]*COLS for _ in range(ROWS)]
        self.startCell = (2, 2)
        self.goalCell  = (ROWS-3, COLS-3)
        self.grid[2][2] = S
        self.grid[ROWS-3][COLS-3] = G

        self.animSteps, self.animIndex = [], 0
        self.pathList, self.pathSet    = [], set()
        self.agentPos, self.agentIndex = None, 0
        self.running, self.searchStart = False, 0.0
        self.animTimer = self.agentTimer = self.obstacleTimer = None

        self.algoVar    = tk.StringVar(value="A*")
        self.heurVar    = tk.StringVar(value="Manhattan")
        self.drawVar    = tk.StringVar(value="wall")
        self.dynamicVar = tk.BooleanVar(value=False)

        self.buildUI()
        self.drawFullGrid()

    # ── UI ────────────────────────────────────────────────────────────────────
    def buildUI(self):
        BG, BG2 = "#1e1e2e", "#11111b"

        def lbl(p, t): tk.Label(p, text=t, bg=BG, fg="#6c7086", font=("Courier",8)).pack(side=tk.LEFT, padx=(8,2))
        def div(p):   tk.Frame(p, bg="#45475a", width=1).pack(side=tk.LEFT, fill=tk.Y, padx=6, pady=3)
        def rb(p, t, var, val, fg):
            tk.Radiobutton(p, text=t, variable=var, value=val, bg=BG, fg=fg,
                           selectcolor="#45475a", activebackground=BG,
                           font=("Courier",9)).pack(side=tk.LEFT, padx=1)

        r1 = tk.Frame(self.root, bg=BG, pady=5, padx=8); r1.pack(fill=tk.X)
        lbl(r1,"ALGORITHM:"); rb(r1,"A*",self.algoVar,"A*","#cdd6f4"); rb(r1,"GBFS",self.algoVar,"GBFS","#cdd6f4"); div(r1)
        lbl(r1,"HEURISTIC:"); rb(r1,"Manhattan",self.heurVar,"Manhattan","#cdd6f4"); rb(r1,"Euclidean",self.heurVar,"Euclidean","#cdd6f4"); div(r1)
        lbl(r1,"DRAW:")
        for mode, col in [("wall","#585b70"),("start","#89b4fa"),("goal","#f38ba8"),("erase","#6c7086")]:
            rb(r1, mode.capitalize(), self.drawVar, mode, col)
        div(r1); lbl(r1,"DYNAMIC:")
        tk.Checkbutton(r1, text="ON", variable=self.dynamicVar, bg=BG, fg="#f9e2af",
                       selectcolor="#45475a", activebackground=BG, font=("Courier",9)).pack(side=tk.LEFT)

        r2 = tk.Frame(self.root, bg=BG2, pady=5, padx=8); r2.pack(fill=tk.X)
        def btn(t, cmd, fg):
            tk.Button(r2, text=t, command=cmd, font=("Courier",9,"bold"), bg=BG, fg=fg,
                      activebackground="#313244", activeforeground=fg,
                      relief=tk.FLAT, padx=12, pady=4, cursor="hand2", bd=0).pack(side=tk.LEFT, padx=3)
        btn("▶  RUN", self.runSearch,"#a6e3a1"); btn("■  STOP",self.stopAll,"#f38ba8")
        btn("⟳  RANDOM MAP",self.randomMap,"#f9e2af"); btn("✕  CLEAR",self.clearGrid,"#6c7086")

        r3 = tk.Frame(self.root, bg="#181825", pady=4, padx=8); r3.pack(fill=tk.X)
        tk.Label(r3, text="METRICS DASHBOARD", bg="#181825", fg="#45475a",
                 font=("Courier",7,"bold")).pack(anchor="w", padx=4, pady=(2,1))
        cardRow = tk.Frame(r3, bg="#181825"); cardRow.pack(fill=tk.X, padx=4, pady=(0,4))
        def card(title, val, color):
            b = tk.Frame(cardRow, bg=BG, padx=14, pady=6); b.pack(side=tk.LEFT, padx=5)
            tk.Label(b, text=title, bg=BG, fg="#6c7086", font=("Courier",7,"bold")).pack(anchor="w")
            v = tk.Label(b, text=val, bg=BG, fg=color, font=("Courier",16,"bold"), width=9, anchor="w"); v.pack(anchor="w")
            return v
        self.mNodes  = card("NODES VISITED","0",   "#f9e2af")
        self.mCost   = card("PATH COST",    "--",  "#a6e3a1")
        self.mTime   = card("EXEC TIME (ms)","--", "#89b4fa")
        self.mStatus = card("STATUS",       "IDLE","#cdd6f4")

        r4 = tk.Frame(self.root, bg=BG2, padx=8, pady=4); r4.pack()
        self.canvas = tk.Canvas(r4, width=COLS*CELL, height=ROWS*CELL,
                                bg=BG2, highlightthickness=2, highlightbackground="#45475a")
        self.canvas.pack()
        self.canvas.bind("<Button-1>",  self.onLeftClick)
        self.canvas.bind("<B1-Motion>", self.onLeftClick)
        self.canvas.bind("<Button-3>",  self.onRightClick)
        self.canvas.bind("<B3-Motion>", self.onRightClick)

        r5 = tk.Frame(self.root, bg=BG2, pady=4, padx=8); r5.pack(fill=tk.X)
        tk.Label(r5, text="LEGEND:", bg=BG2, fg="#6c7086", font=("Courier",7,"bold")).pack(side=tk.LEFT, padx=(0,6))
        for name, col in [("Frontier",COLOR[F]),("Visited",COLOR[V]),("Path",COLOR[P]),
                          ("Agent",COLOR[A]),("Start(S)",COLOR[S]),("Goal(G)",COLOR[G]),("Wall",COLOR[W])]:
            f = tk.Frame(r5, bg=BG2); f.pack(side=tk.LEFT, padx=4)
            tk.Canvas(f, width=12, height=12, bg=col, highlightthickness=0).pack(side=tk.LEFT)
            tk.Label(f, text=" "+name, bg=BG2, fg="#cdd6f4", font=("Courier",7)).pack(side=tk.LEFT)

    # ── Drawing ───────────────────────────────────────────────────────────────
    def drawFullGrid(self):
        self.canvas.delete("all")
        for i in range(ROWS+1): self.canvas.create_line(0,i*CELL,COLS*CELL,i*CELL,fill="#181825")
        for i in range(COLS+1): self.canvas.create_line(i*CELL,0,i*CELL,ROWS*CELL,fill="#181825")
        for r in range(ROWS):
            for c in range(COLS): self.drawCell(r, c)

    def drawCell(self, r, c):
        x0,y0 = c*CELL+1, r*CELL+1
        x1,y1 = x0+CELL-2, y0+CELL-2
        cx,cy = (x0+x1)//2, (y0+y1)//2
        tag = f"cell{r}x{c}"
        self.canvas.delete(tag)
        self.canvas.create_rectangle(x0,y0,x1,y1, fill=COLOR.get(self.grid[r][c],COLOR[E]), outline="", tags=tag)
        label = "S" if (r,c)==self.startCell else "G" if (r,c)==self.goalCell else "●" if (r,c)==self.agentPos else None
        if label: self.canvas.create_text(cx,cy, text=label, fill="#11111b", font=("Courier",10,"bold"), tags=tag)

    # ── Mouse ─────────────────────────────────────────────────────────────────
    def getCell(self, event):
        r, c = event.y//CELL, event.x//CELL
        return (r,c) if 0<=r<ROWS and 0<=c<COLS else (None,None)

    def onLeftClick(self, event):
        r, c = self.getCell(event)
        if r is None: return
        mode = self.drawVar.get()
        if mode == "wall":
            if (r,c) in (self.startCell, self.goalCell): return
            self.grid[r][c] = W; self.drawCell(r,c)
            self.checkWallOnPath(r, c); return
        elif mode == "erase":
            if (r,c) in (self.startCell, self.goalCell): return
            self.grid[r][c] = E
        elif mode == "start":
            old = self.startCell; self.grid[old[0]][old[1]] = E
            self.startCell = (r,c); self.grid[r][c] = S; self.drawCell(*old)
        elif mode == "goal":
            old = self.goalCell; self.grid[old[0]][old[1]] = E
            self.goalCell = (r,c); self.grid[r][c] = G; self.drawCell(*old)
        self.drawCell(r, c)

    def onRightClick(self, event):
        r, c = self.getCell(event)
        if r is None or (r,c) in (self.startCell, self.goalCell): return
        self.grid[r][c] = E; self.drawCell(r, c)

    # ── Helpers ───────────────────────────────────────────────────────────────
    def setStatus(self, text, color="#cdd6f4"): self.mStatus.config(text=text, fg=color)

    def resetMetrics(self):
        self.mNodes.config(text="0"); self.mCost.config(text="--")
        self.mTime.config(text="--"); self.setStatus("IDLE","#6c7086")

    def stopAll(self):
        self.running = False
        for t in [self.animTimer, self.agentTimer, self.obstacleTimer]:
            if t: self.root.after_cancel(t)
        self.animTimer = self.agentTimer = self.obstacleTimer = None
        self.setStatus("STOPPED","#f38ba8")

    def clearGrid(self):
        self.stopAll()
        self.grid = [[E]*COLS for _ in range(ROWS)]
        self.grid[self.startCell[0]][self.startCell[1]] = S
        self.grid[self.goalCell[0]][self.goalCell[1]]   = G
        self.agentPos = None; self.pathList = []; self.pathSet = set()
        self.resetMetrics(); self.drawFullGrid()

    def randomMap(self):
        self.stopAll()
        self.agentPos = None; self.pathList = []; self.pathSet = set()
        self.grid = [[E]*COLS for _ in range(ROWS)]
        for r in range(ROWS):
            for c in range(COLS):
                if (r,c) not in (self.startCell,self.goalCell) and random.random()<0.28:
                    self.grid[r][c] = W
        self.grid[self.startCell[0]][self.startCell[1]] = S
        self.grid[self.goalCell[0]][self.goalCell[1]]   = G
        self.resetMetrics(); self.drawFullGrid(); self.setStatus("MAP READY","#f9e2af")

    def getHeuristic(self): return manhattanDist if self.heurVar.get()=="Manhattan" else euclideanDist
    def getAlgorithm(self): return runAstar if self.algoVar.get()=="A*" else runGbfs

    # ── Run search ────────────────────────────────────────────────────────────
    def runSearch(self):
        self.stopAll()
        for r in range(ROWS):
            for c in range(COLS):
                if self.grid[r][c] in (F,V,P,A): self.grid[r][c] = E
        self.agentPos = None; self.pathList = []; self.pathSet = set(); self.agentIndex = 0
        self.drawFullGrid(); self.resetMetrics()
        self.searchStart = time.perf_counter()
        self.animSteps = self.getAlgorithm()(self.grid, self.startCell, self.goalCell, self.getHeuristic())
        if not self.animSteps: self.setStatus("NO PATH","#f38ba8"); return
        self.animIndex = 0; self.running = True
        self.setStatus("SEARCHING...","#f9e2af"); self.animateStep()

    # ── Animation ─────────────────────────────────────────────────────────────
    def animateStep(self):
        if not self.running or self.animIndex >= len(self.animSteps): return
        frontier, visited, path = self.animSteps[self.animIndex]; self.animIndex += 1
        for r in range(ROWS):
            for c in range(COLS):
                if (r,c) in (self.startCell,self.goalCell) or self.grid[r][c]==W: continue
                new = F if (r,c) in frontier else V if (r,c) in visited else E
                if new != self.grid[r][c]: self.grid[r][c] = new; self.drawCell(r,c)
        self.mNodes.config(text=str(len(visited)))
        if path is not None:
            self.mTime.config(text=f"{(time.perf_counter()-self.searchStart)*1000:.2f}")
            if not path: self.setStatus("NO PATH FOUND","#f38ba8"); self.running=False; return
            for r,c in path:
                if (r,c) not in (self.startCell,self.goalCell): self.grid[r][c]=P; self.drawCell(r,c)
            self.pathList=path; self.pathSet=set(path); self.agentIndex=0; self.agentPos=self.startCell
            self.mCost.config(text=str(len(path)-1)); self.setStatus("PATH FOUND!","#a6e3a1")
            self.agentTimer    = self.root.after(350, self.moveAgent)
            self.obstacleTimer = self.root.after(700, self.spawnObstacle)
            return
        self.animTimer = self.root.after(55, self.animateStep)

    # ── Agent movement ────────────────────────────────────────────────────────
    def moveAgent(self):
        if not self.running: return
        if self.agentIndex >= len(self.pathList)-1: self.setStatus("DONE ✓","#a6e3a1"); return
        nextIdx = self.agentIndex + 1
        nr, nc  = self.pathList[nextIdx]
        if self.grid[nr][nc] == W: self.doReplan(); return
        if self.agentPos and self.agentPos not in (self.startCell,self.goalCell):
            pr,pc = self.agentPos; self.grid[pr][pc]=P; self.drawCell(pr,pc)
        self.agentIndex=nextIdx; self.agentPos=(nr,nc)
        self.grid[nr][nc]=A; self.drawCell(nr,nc)
        self.agentTimer = self.root.after(100, self.moveAgent)

    # ── Dynamic obstacles ─────────────────────────────────────────────────────
    def spawnObstacle(self):
        if not self.running: return
        if self.dynamicVar.get():
            for _ in range(50):
                r,c = random.randint(0,ROWS-1), random.randint(0,COLS-1)
                if (r,c) in (self.startCell,self.goalCell,self.agentPos): continue
                if self.grid[r][c] not in (E,P,F,V): continue
                self.grid[r][c]=W; self.drawCell(r,c); self.checkWallOnPath(r,c); break
        self.obstacleTimer = self.root.after(600, self.spawnObstacle)

    # ── Wall-on-path check → triggers replan ──────────────────────────────────
    def checkWallOnPath(self, r, c):
        if not self.pathList or (r,c) not in self.pathList[self.agentIndex:]: return
        for t in [self.agentTimer, self.animTimer]:
            if t: self.root.after_cancel(t)
        self.agentTimer = self.animTimer = None
        self.running = True; self.doReplan()

    # ── Replanning ────────────────────────────────────────────────────────────
    def doReplan(self):
        self.setStatus("REPLANNING...","#f9e2af")
        cur = self.agentPos or self.startCell
        for r,c in self.pathList:
            if (r,c) not in (self.startCell,self.goalCell,cur) and self.grid[r][c]==P:
                self.grid[r][c]=E; self.drawCell(r,c)
        t0 = time.perf_counter()
        newPath = quickPath(self.grid, cur, self.goalCell, self.getHeuristic(), self.getAlgorithm())
        self.mTime.config(text=f"{(time.perf_counter()-t0)*1000:.2f}")
        if not newPath: self.setStatus("NO PATH FOUND","#f38ba8"); self.running=False; return
        for r,c in newPath:
            if (r,c) not in (self.startCell,self.goalCell,cur): self.grid[r][c]=P; self.drawCell(r,c)
        self.pathList=newPath; self.pathSet=set(newPath); self.agentIndex=0; self.agentPos=cur
        self.mCost.config(text=str(len(newPath)-1)); self.setStatus("REPLANNED!","#cba6f7")
        self.agentTimer = self.root.after(150, self.moveAgent)

if __name__ == "__main__":
    root = tk.Tk(); App(root); root.mainloop()